# Fisher-KPP Geo-Spectral Forward PINN Lab

Objective: run a Colab-ready Fisher-KPP forward PINN experiment. By default this notebook uses the Geo-Spectral Causal Adaptive gPINN profile on the same Korea pine-wilt style problem setup: diffusion-reaction Fisher-KPP, learnable `D` and `r`, no advection term, square-domain valid collocation, known initial-condition loss, residual curriculum, and adaptive relative loss balancing.

Set `USE_GEO_SPECTRAL_FORWARD = False` and `USE_KOREA_PINE_STYLE = True` in the configuration cell to run the simpler forward baseline.


In [ ]:
%matplotlib inline

from __future__ import annotations

import base64
import io
import json
import sys
import zipfile
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image, Markdown, display

# Colab/self-contained bootstrap -------------------------------------------------
# If the local package is missing, this cell reconstructs the project files from an
# embedded source archive. That makes the notebook runnable after uploading only the
# .ipynb file to Google Colab.
_EMBEDDED_PROJECT_ZIP_B64 = """
...
"""


def _find_project_root() -> Path | None:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "fisher_origin_lab").exists():
            return candidate
    return None


def _bootstrap_embedded_project() -> Path:
    try:
        import google.colab  # type: ignore  # noqa: F401
        target = Path("/content/fisher-kpp-origin-lab")
    except Exception:
        target = Path.cwd().resolve() / "fisher-kpp-origin-lab"
    target.mkdir(parents=True, exist_ok=True)
    raw = base64.b64decode("".join(_EMBEDDED_PROJECT_ZIP_B64.split()))
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        zf.extractall(target)
    return target.resolve()

PROJECT_ROOT = _find_project_root()
if PROJECT_ROOT is None:
    PROJECT_ROOT = _bootstrap_embedded_project()

if not (PROJECT_ROOT / "fisher_origin_lab").exists():
    raise RuntimeError(f"Could not locate or bootstrap fisher_origin_lab under {PROJECT_ROOT}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"project root: {PROJECT_ROOT}")
print(f"torch: {torch.__version__} | cuda available: {torch.cuda.is_available()}")


## Plan

1. Select the Korea pine-wilt compatible problem profile or the inverse-origin profile.
2. Preview the generated truth and sensor locations.
3. Train the PINN with causal weighting, residual curriculum, adaptive relative loss balancing, and held-out observation validation.
4. Inspect reconstruction quality, learned physics, RK4 accuracy, and stabilizer diagnostics.


In [ ]:
from fisher_origin_lab.config import (
    ExperimentConfig,
    LossWeights,
    ModelConfig,
    ObservationConfig,
)
from fisher_origin_lab.simulate import forward_fisher_kpp, sample_observations, truth_field_at
from fisher_origin_lab.train import run_experiment

# Notebook defaults are chosen to finish quickly on Colab while exercising the full pipeline.
USE_GEO_SPECTRAL_FORWARD = True
USE_KOREA_PINE_STYLE = False
RUN_NAME = "notebook_geo_spectral_forward" if USE_GEO_SPECTRAL_FORWARD else "notebook_korea_pine_style"
QUICK = True
EPOCHS = 60
ENSEMBLE = 1
RUN_DIFFERENTIABLE_BASELINE = False
BASELINE_EPOCHS = 60
BASE_SEED = 7

base_cfg = ExperimentConfig(
    observations=ObservationConfig(samples_per_frame=500, noise_std=0.02, focus_fraction=0.5),
    model=ModelConfig(learn_drift=False, learn_diffusion=False, learn_reaction=False),
    weights=LossWeights(gradient=0.01),
    ensemble=ENSEMBLE,
    base_seed=BASE_SEED,
    out_dir=PROJECT_ROOT / "runs" / RUN_NAME,
    run_classical_baseline=RUN_DIFFERENTIABLE_BASELINE,
    baseline_epochs=BASELINE_EPOCHS,
)

if USE_GEO_SPECTRAL_FORWARD:
    base_cfg = base_cfg.geo_spectral_forward()
elif USE_KOREA_PINE_STYLE:
    base_cfg = base_cfg.korea_pine_style()

cfg = base_cfg.quick() if QUICK else base_cfg
cfg = replace(
    cfg,
    out_dir=PROJECT_ROOT / "runs" / RUN_NAME,
    ensemble=ENSEMBLE,
    run_classical_baseline=RUN_DIFFERENTIABLE_BASELINE,
    baseline_epochs=BASELINE_EPOCHS,
    train=replace(cfg.train, epochs=EPOCHS, print_every=max(1, EPOCHS // 4)),
)

print(json.dumps(cfg.to_dict(), indent=2, default=str))


## Data Preview

The synthetic observation design mixes uniform sensors with front-focused sensors. This avoids a degenerate dataset where most observations are nearly zero background.

In [ ]:
rng = np.random.default_rng(cfg.base_seed)
truth = forward_fisher_kpp(cfg.domain, cfg.pde, cfg.seed)
observations = sample_observations(truth, cfg.domain, cfg.observations, rng)

print(f"truth fields: {truth.fields.shape}")
print(f"observations: {observations.xyt.shape}, values: {observations.values.shape}")
print(f"value range: [{observations.values.min():.3f}, {observations.values.max():.3f}]")

times = [0.0, cfg.observations.start_time, cfg.domain.t_end]
fig, axes = plt.subplots(1, len(times), figsize=(12, 3.5), constrained_layout=True)
for ax, t in zip(axes, times):
    xs, field = truth_field_at(truth, t, n=96)
    ax.imshow(field.T, origin="lower", extent=[0, cfg.domain.box, 0, cfg.domain.box], cmap="magma", vmin=0, vmax=1)
    ax.plot(cfg.seed.center_x, cfg.seed.center_y, marker="*", color="cyan", markersize=12, markeredgecolor="white")
    if t == cfg.domain.t_end:
        latest = np.isclose(observations.xyt[:, 2], cfg.domain.t_end)
        ax.scatter(observations.xyt[latest, 0], observations.xyt[latest, 1], s=5, c="white", alpha=0.35, linewidths=0)
    ax.set_title(f"truth t={t:.2f}")
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

## Run The Forward PINN Experiment

This cell writes metrics and visual diagnostics to `cfg.out_dir`. The run exports observation coverage, reconstruction/error panels, space-time error trends, residual/front maps, RK4 comparison, adaptive loss multipliers, and training diagnostics.


In [ ]:
metrics = run_experiment(cfg)
metrics_path = cfg.out_dir / "metrics.json"
figure_paths = [Path(path) for path in metrics.get("figures", [])]

print(f"metrics: {metrics_path}")
for path in figure_paths:
    print(f"figure : {path}")


## Metrics

For the Geo-Spectral forward profile, the primary checks are reconstruction error, train/validation observation MSE, known initial-condition loss, learned `D/r`, boundary loss, front-local residual-gradient loss, residual-curriculum exponent, and adaptive loss multipliers. Origin rows remain diagnostic because source-envelope inverse inference is intentionally off in this profile.


In [ ]:
def fmt_center(center):
    if center is None:
        return "-"
    return f"({center[0]:.3f}, {center[1]:.3f})"

rows = []
for baseline in metrics["baselines"]:
    err = baseline["error"]
    rows.append([baseline["name"], fmt_center(baseline["center"]), "-" if err is None else f"{err:.4f}"])
method_name = "Geo-Spectral forward PINN" if cfg.model.use_geo_features else "Korea-style forward PINN"
rows.append([method_name, fmt_center(metrics["best_origin"]), f"{metrics['best_origin_error']:.4f}"])

md = "| method | center | origin diagnostic |\n|---|---:|---:|\n"
for name, center, err in rows:
    md += f"| {name} | {center} | {err} |\n"
display(Markdown(md))

print("final-time relative L2:", round(metrics["final_time_relative_l2"], 4))
print("train observation MSE:", round(metrics["train_observation_mse"], 6))
print("validation observation MSE:", None if metrics["validation_observation_mse"] is None else round(metrics["validation_observation_mse"], 6))
print("learned physics:", {k: round(v, 6) for k, v in metrics["runs"][0]["physics"].items() if k in ["diffusion", "reaction", "velocity_x", "velocity_y"]})
print("geo:", cfg.geo)
print("IC/boundary/front/sparse weights:", {k: getattr(cfg.weights, k) for k in ["initial_condition", "boundary", "front_pde_alpha", "front_pde_gradient", "front_gradient", "sparse"]})
print("adaptive/curriculum:", {"adaptive_loss_balancing": cfg.train.adaptive_loss_balancing, "residual_curriculum_epochs": cfg.train.residual_curriculum_epochs, "residual_exponent": (cfg.train.residual_weight_exponent_start, cfg.train.residual_weight_exponent_end)})


## PINN vs RK4 Accuracy

The RK4 baseline here is the RK4 time integrator adapted to the same 2D square-domain Fisher-KPP problem, Gaussian seed, and Neumann boundary condition as the PINN experiment. The Geo-Spectral PINN receives the same known Gaussian initial condition through an explicit `initial_condition` loss, then uses residual curriculum and adaptive loss balancing to reduce the multi-objective training imbalance.


In [ ]:
def fmt_metric_value(value):
    if value is None:
        return "-"
    return f"{float(value):.4e}"


def display_accuracy_comparison(run_metrics, title="quick run"):
    rows = [
        ("PINN final relative L2 vs reference", run_metrics.get("pinn_final_time_relative_l2", run_metrics.get("final_time_relative_l2"))),
        ("RK4 final relative L2 vs reference", run_metrics.get("rk4_final_time_relative_l2")),
        ("PINN/RK4 final relative L2", run_metrics.get("pinn_vs_rk4_final_relative_l2")),
        ("PINN validation observation MSE", run_metrics.get("validation_observation_mse")),
        ("RK4 validation observation MSE", run_metrics.get("rk4_validation_observation_mse")),
        ("RK4 runtime (sec)", run_metrics.get("rk4_runtime_sec")),
    ]
    md = f"### {title}\n\n| metric | value |\n|---|---:|\n"
    for name, value in rows:
        md += f"| {name} | {fmt_metric_value(value)} |\n"
    display(Markdown(md))

    comparison_path = next(
        (Path(path) for path in run_metrics.get("figures", []) if Path(path).name == "pinn_vs_rk4_comparison.png"),
        None,
    )
    if comparison_path is not None and comparison_path.exists():
        display(Image(filename=str(comparison_path)))


display_accuracy_comparison(metrics, title="quick run")


## Diagnostic Figures

The RK4 comparison figure is shown above; the remaining figures inspect observations, reconstruction quality, residual/front weighting, adaptive loss balancing, residual curriculum, and training dynamics.


In [ ]:
for path in figure_paths:
    if path.name == "pinn_vs_rk4_comparison.png":
        continue
    if path.exists():
        display(Markdown(f"### {path.name}"))
        display(Image(filename=str(path)))


## Training Curves

The quick configuration is mainly a pipeline sanity check. The curves below expose known-IC loss, residual curriculum, and adaptive multipliers so unstable loss competition is visible before running the full experiment.


In [ ]:
history = metrics["runs"][0]["history"]
epochs = [row["epoch"] for row in history]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), constrained_layout=True)
axes[0].plot(epochs, [row["total"] for row in history], marker="o", label="total")
axes[0].plot(epochs, [row["data"] for row in history], marker="o", label="data")
axes[0].plot(epochs, [row["pde"] for row in history], marker="o", label="pde")
axes[0].plot(epochs, [row.get("ic", 0.0) for row in history], marker="o", label="known IC")
axes[0].set_yscale("log")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.25)

if cfg.model.use_source_envelope:
    axes[1].plot(epochs, [row["origin_error"] for row in history], marker="o", color="tab:green", label="origin error")
else:
    axes[1].plot(epochs, [row.get("bc", 0.0) for row in history], marker="o", label="bc")
    axes[1].plot(epochs, [row.get("front_grad", 0.0) for row in history], marker="o", label="front gPINN")
    axes[1].plot(epochs, [row.get("front_weight_mean", 1.0) for row in history], marker="o", label="front weight")
    axes[1].plot(epochs, [row.get("residual_exponent", 0.0) for row in history], marker="o", label="residual exponent")
    axes[1].plot(epochs, [row.get("sparse", 0.0) for row in history], marker="o", label="sparse L1")
axes[1].set_yscale("log")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("geo/front diagnostics")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.25)

adaptive_keys = ["aw_data", "aw_pde", "aw_ic", "aw_bc", "aw_front_grad", "aw_sparse"]
plotted = False
for key in adaptive_keys:
    if any(key in row for row in history):
        axes[2].plot(epochs, [row.get(key, np.nan) for row in history], marker="o", label=key.replace("aw_", ""))
        plotted = True
if plotted:
    axes[2].set_ylabel("adaptive multiplier")
    axes[2].legend(fontsize=8, ncol=2)
else:
    axes[2].plot(epochs, [row.get("elapsed_sec", 0.0) for row in history], marker="o", color="tab:gray")
    axes[2].set_ylabel("elapsed sec")
axes[2].set_xlabel("epoch")
axes[2].grid(True, alpha=0.25)
plt.show()


## Optional Full Run

Set `RUN_FULL = True` for a stronger experiment. The full run writes the same diagnostic figure set and also displays the same PINN-vs-RK4 accuracy table and comparison figure, so visual comparison remains consistent across smoke, quick, and full settings.


In [ ]:
RUN_FULL = False

if RUN_FULL:
    full_cfg = replace(
        base_cfg,
        out_dir=PROJECT_ROOT / "runs" / ("notebook_geo_spectral_full" if USE_GEO_SPECTRAL_FORWARD else "notebook_forward_full"),
        ensemble=1,
        run_classical_baseline=True,
        baseline_epochs=250,
        train=replace(base_cfg.train, epochs=1200, print_every=100),
    )
    full_metrics = run_experiment(full_cfg)
    full_figure_paths = [Path(path) for path in full_metrics.get("figures", [])]
    display_accuracy_comparison(full_metrics, title="full run")
    for path in full_figure_paths:
        if path.name == "pinn_vs_rk4_comparison.png":
            continue
        if path.exists():
            display(Markdown(f"### full run: {path.name}"))
            display(Image(filename=str(path)))
else:
    print("RUN_FULL is False. Flip it to True when you want the slower validation run.")


## Optional Ablation Matrix

Run this after the quick experiment when you want to test whether the result depends on drift-corrected warm starts or source anchoring. The default here is a very small smoke matrix; switch to `--preset quick --case-set core --seeds 7,8,9` for a more useful comparison.

In [ ]:
RUN_ABLATION = False

if RUN_ABLATION:
    import subprocess

    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "run_ablation.py"),
        "--preset", "smoke",
        "--case-set", "anchor",
        "--seeds", "7",
        "--out-dir", str(PROJECT_ROOT / "runs" / "notebook_ablation_smoke"),
    ]
    subprocess.run(cmd, check=True)
    display(Image(filename=str(PROJECT_ROOT / "runs" / "notebook_ablation_smoke" / "summary.png")))
else:
    print("RUN_ABLATION is False. Flip it to True for the optional ablation smoke matrix.")

## Notes For Reporting

- Do not claim this is a PINN-only capability. It is a PDE-constrained inverse/forward comparison problem.
- Report the observation-only drift-corrected centroid baseline alongside the PINN result when source inference is enabled.
- Report `validation_observation_mse`; training-only data fit is not enough.
- Report `pinn_final_time_relative_l2`, `rk4_final_time_relative_l2`, and `pinn_vs_rk4_final_relative_l2` together so the neural and numerical solvers are compared on the same PDE setting.
- Report whether known IC, residual curriculum, and adaptive loss balancing were enabled; these materially change the training objective.
- Use `shooting_prefit` and `known_drift_no_shooting` ablations to separate method contribution from warm-start quality.
- Use ensemble spread as an uncertainty indicator when `ENSEMBLE > 1`.
- Treat the quick run as a smoke test; use the full run before drawing conclusions about field reconstruction.
